**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 14: 데이터 수집/증강/정제 파이프라인

**Part 3: 파인튜닝 기초** | GPU: No

---

### 📋 학습 목표
- 🎯 파인튜닝 데이터의 주요 형식(Alpaca, ShareGPT, ChatML)을 이해한다
- 🎯 데이터 수집, 정제, 증강의 전체 파이프라인을 파악한다
- 🎯 실제 데이터셋을 로드하고 변환하는 실습을 수행한다
- 🎯 HuggingFace datasets 라이브러리 활용법을 익힌다

### ⏱️ 예상 소요 시간: 90분

In [1]:
# 환경 설정
import json
import os
import random
from collections import Counter

print("✅ Session 14: 데이터 수집/증강/정제 파이프라인")
print("📌 이 노트북은 GPU가 필요하지 않습니다.")
print()

# 작업 디렉토리 확인
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "samples")
os.makedirs(DATA_DIR, exist_ok=True)
print(f"📁 데이터 디렉토리: {DATA_DIR}")

✅ Session 14: 데이터 수집/증강/정제 파이프라인
📌 이 노트북은 GPU가 필요하지 않습니다.

📁 데이터 디렉토리: /home/yskim/LLM_Advanced/data/samples


---
## 🎯 1. 파인튜닝 데이터 파이프라인 개요

### 전체 파이프라인

```
📥 데이터 수집 (Collection)
    ↓
🔍 데이터 정제 (Cleaning)
    ↓
📐 데이터 형식 변환 (Formatting)
    ↓
📊 데이터 증강 (Augmentation) [선택]
    ↓
✅ 품질 검증 (Validation)
    ↓
🚀 학습 데이터 준비 완료
```

### 데이터 품질이 중요한 이유

- 📌 **"Garbage In, Garbage Out"** - 나쁜 데이터로는 좋은 모델을 만들 수 없음
- 📌 LIMA 논문 (2023): **1,000개의 고품질 데이터**로 GPT-4 수준의 응답 가능
- 📌 데이터 양보다 **데이터 품질**이 더 중요
- 📌 현업에서 파인튜닝 시간의 70%는 데이터 준비에 사용

---
## 1️⃣ 2. 데이터 형식: Alpaca vs ShareGPT vs ChatML

### Alpaca 형식
Stanford Alpaca 프로젝트에서 사용한 형식으로, 가장 단순합니다.

```json
{
    "instruction": "지시 내용",
    "input": "입력 데이터 (선택)",
    "output": "기대 출력"
}
```

### ShareGPT 형식
멀티턴 대화를 지원하는 형식입니다.

```json
{
    "conversations": [
        {"from": "human", "value": "사용자 메시지"},
        {"from": "gpt", "value": "AI 응답"},
        {"from": "human", "value": "후속 질문"},
        {"from": "gpt", "value": "후속 응답"}
    ]
}
```

### ChatML 형식
OpenAI의 Chat Completion API와 호환되는 형식입니다.

```json
{
    "messages": [
        {"role": "system", "content": "시스템 프롬프트"},
        {"role": "user", "content": "사용자 메시지"},
        {"role": "assistant", "content": "AI 응답"}
    ]
}
```

In [2]:
# 각 형식 예시 생성 및 비교
print("📊 데이터 형식 비교")
print("=" * 60)

# 1. Alpaca 형식
alpaca_example = {
    "instruction": "다음 문장의 감성을 분석하세요.",
    "input": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다.",
    "output": "이 문장은 혼합된 감성을 보여줍니다. 음식의 맛에 대해서는 긍정적('정말 맛있었지만')이나, 서비스 속도에 대해서는 부정적('너무 느렸습니다')입니다. 전반적으로 중립에서 약간 긍정적인 감성으로 판단됩니다."
}

print("\n📋 1. Alpaca 형식:")
print(json.dumps(alpaca_example, ensure_ascii=False, indent=2))

# 2. ShareGPT 형식
sharegpt_example = {
    "conversations": [
        {"from": "human", "value": "파이썬에서 리스트를 정렬하는 방법을 알려주세요."},
        {"from": "gpt", "value": "파이썬에서 리스트를 정렬하는 방법은 크게 두 가지입니다:\n\n1. `sort()` 메서드: 원본 리스트를 직접 정렬\n2. `sorted()` 함수: 새로운 정렬된 리스트 반환"},
        {"from": "human", "value": "역순으로 정렬하려면요?"},
        {"from": "gpt", "value": "`reverse=True` 파라미터를 사용하면 됩니다:\n```python\nmy_list.sort(reverse=True)\n# 또는\nsorted_list = sorted(my_list, reverse=True)\n```"}
    ]
}

print("\n📋 2. ShareGPT 형식:")
print(json.dumps(sharegpt_example, ensure_ascii=False, indent=2))

# 3. ChatML 형식
chatml_example = {
    "messages": [
        {"role": "system", "content": "당신은 한국어 감성 분석 전문가입니다."},
        {"role": "user", "content": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다."},
        {"role": "assistant", "content": "감성 분석 결과:\n- 음식: 긍정 (맛있었지만)\n- 서비스: 부정 (너무 느렸습니다)\n- 종합: 혼합 (약간 긍정)"}
    ]
}

print("\n📋 3. ChatML 형식:")
print(json.dumps(chatml_example, ensure_ascii=False, indent=2))

📊 데이터 형식 비교

📋 1. Alpaca 형식:
{
  "instruction": "다음 문장의 감성을 분석하세요.",
  "input": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다.",
  "output": "이 문장은 혼합된 감성을 보여줍니다. 음식의 맛에 대해서는 긍정적('정말 맛있었지만')이나, 서비스 속도에 대해서는 부정적('너무 느렸습니다')입니다. 전반적으로 중립에서 약간 긍정적인 감성으로 판단됩니다."
}

📋 2. ShareGPT 형식:
{
  "conversations": [
    {
      "from": "human",
      "value": "파이썬에서 리스트를 정렬하는 방법을 알려주세요."
    },
    {
      "from": "gpt",
      "value": "파이썬에서 리스트를 정렬하는 방법은 크게 두 가지입니다:\n\n1. `sort()` 메서드: 원본 리스트를 직접 정렬\n2. `sorted()` 함수: 새로운 정렬된 리스트 반환"
    },
    {
      "from": "human",
      "value": "역순으로 정렬하려면요?"
    },
    {
      "from": "gpt",
      "value": "`reverse=True` 파라미터를 사용하면 됩니다:\n```python\nmy_list.sort(reverse=True)\n# 또는\nsorted_list = sorted(my_list, reverse=True)\n```"
    }
  ]
}

📋 3. ChatML 형식:
{
  "messages": [
    {
      "role": "system",
      "content": "당신은 한국어 감성 분석 전문가입니다."
    },
    {
      "role": "user",
      "content": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다."
    },
    {
      "role": "assist

In [3]:
# 형식 간 변환 함수
print("🔄 데이터 형식 변환 함수")
print("=" * 60)

def alpaca_to_chatml(alpaca_data):
    """Alpaca → ChatML 변환"""
    messages = []
    
    # user 메시지 구성
    user_content = alpaca_data["instruction"]
    if alpaca_data.get("input", "").strip():
        user_content += f"\n\n{alpaca_data['input']}"
    
    messages.append({"role": "user", "content": user_content})
    messages.append({"role": "assistant", "content": alpaca_data["output"]})
    
    return {"messages": messages}

def sharegpt_to_chatml(sharegpt_data):
    """ShareGPT → ChatML 변환"""
    role_map = {"human": "user", "gpt": "assistant", "system": "system"}
    messages = []
    
    for conv in sharegpt_data["conversations"]:
        role = role_map.get(conv["from"], conv["from"])
        messages.append({"role": role, "content": conv["value"]})
    
    return {"messages": messages}

def chatml_to_alpaca(chatml_data):
    """ChatML → Alpaca 변환 (단일 턴만)"""
    messages = chatml_data["messages"]
    user_msg = ""
    assistant_msg = ""
    
    for msg in messages:
        if msg["role"] == "user":
            user_msg = msg["content"]
        elif msg["role"] == "assistant":
            assistant_msg = msg["content"]
    
    return {
        "instruction": user_msg,
        "input": "",
        "output": assistant_msg
    }

# 변환 테스트
print("\n🔄 Alpaca → ChatML 변환:")
converted = alpaca_to_chatml(alpaca_example)
print(json.dumps(converted, ensure_ascii=False, indent=2))

print("\n🔄 ShareGPT → ChatML 변환:")
converted2 = sharegpt_to_chatml(sharegpt_example)
print(json.dumps(converted2, ensure_ascii=False, indent=2))

print("\n💡 실무에서는 ChatML 형식으로 통일하는 것을 권장합니다.")
print("   (대부분의 최신 모델이 ChatML 형식을 지원)")

🔄 데이터 형식 변환 함수

🔄 Alpaca → ChatML 변환:
{
  "messages": [
    {
      "role": "user",
      "content": "다음 문장의 감성을 분석하세요.\n\n이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다."
    },
    {
      "role": "assistant",
      "content": "이 문장은 혼합된 감성을 보여줍니다. 음식의 맛에 대해서는 긍정적('정말 맛있었지만')이나, 서비스 속도에 대해서는 부정적('너무 느렸습니다')입니다. 전반적으로 중립에서 약간 긍정적인 감성으로 판단됩니다."
    }
  ]
}

🔄 ShareGPT → ChatML 변환:
{
  "messages": [
    {
      "role": "user",
      "content": "파이썬에서 리스트를 정렬하는 방법을 알려주세요."
    },
    {
      "role": "assistant",
      "content": "파이썬에서 리스트를 정렬하는 방법은 크게 두 가지입니다:\n\n1. `sort()` 메서드: 원본 리스트를 직접 정렬\n2. `sorted()` 함수: 새로운 정렬된 리스트 반환"
    },
    {
      "role": "user",
      "content": "역순으로 정렬하려면요?"
    },
    {
      "role": "assistant",
      "content": "`reverse=True` 파라미터를 사용하면 됩니다:\n```python\nmy_list.sort(reverse=True)\n# 또는\nsorted_list = sorted(my_list, reverse=True)\n```"
    }
  ]
}

💡 실무에서는 ChatML 형식으로 통일하는 것을 권장합니다.
   (대부분의 최신 모델이 ChatML 형식을 지원)


---
## 2️⃣ 3. 데이터 수집 방법

### 데이터 소스 분류

| 소스 | 장점 | 단점 | 예시 |
|------|------|------|------|
| **공개 데이터셋** | 무료, 검증됨 | 도메인 제한 | HuggingFace Hub, KorQuAD |
| **합성 데이터** | 대량 생성 가능 | 품질 검증 필요 | GPT-4로 생성 |
| **크롤링** | 도메인 특화 | 저작권 문제 | 뉴스, 블로그 |
| **직접 제작** | 최고 품질 | 비용/시간 | 전문가 작성 |

### 한국어 공개 데이터셋

| 데이터셋 | 크기 | 용도 |
|---------|------|------|
| **KoAlpaca** | ~52K | 범용 Instruction |
| **KOpen-platypus** | ~25K | 논리/수학 |
| **KorQuAD 2.0** | ~100K | 질의응답 |
| **NSMC** | ~200K | 감성 분석 |
| **AI Hub** | 다양 | 번역, 요약, QA 등 |

In [4]:
# 샘플 데이터 생성 (실습용)
print("📝 실습용 샘플 데이터 생성")
print("=" * 60)

# Alpaca 형식 한국어 샘플 데이터 생성
sample_data = [
    {
        "instruction": "다음 문장을 영어로 번역하세요.",
        "input": "오늘 날씨가 매우 좋습니다.",
        "output": "The weather is very nice today."
    },
    {
        "instruction": "다음 텍스트의 감성을 분석하세요.",
        "input": "이 영화는 정말 재미있었어요! 배우들의 연기도 훌륭했습니다.",
        "output": "긍정적인 감성입니다. '재미있었어요', '훌륭했습니다' 등의 긍정적 표현이 사용되었습니다."
    },
    {
        "instruction": "주어진 키워드로 짧은 이야기를 작성하세요.",
        "input": "고양이, 비, 카페",
        "output": "비가 오는 오후, 작은 카페 창가에 고양이 한 마리가 앉아 있었습니다. 빗소리를 들으며 따뜻한 카페 안에서 졸고 있는 고양이의 모습이 평화로웠습니다. 카페 주인은 조용히 우유 한 잔을 고양이 옆에 놓아주었습니다."
    },
    {
        "instruction": "파이썬에서 리스트 컴프리헨션의 사용법을 설명하세요.",
        "input": "",
        "output": "리스트 컴프리헨션은 파이썬에서 리스트를 간결하게 생성하는 방법입니다.\n\n기본 문법: [expression for item in iterable if condition]\n\n예시:\n1. 1~10의 제곱: [x**2 for x in range(1, 11)]\n2. 짝수만 필터링: [x for x in range(20) if x % 2 == 0]\n3. 문자열 처리: [s.upper() for s in words if len(s) > 3]"
    },
    {
        "instruction": "다음 문장을 3줄로 요약하세요.",
        "input": "인공지능(AI)은 1950년대 앨런 튜링의 연구에서 시작되어 오랜 역사를 가지고 있습니다. 초기에는 규칙 기반 시스템이 주류였으나, 2010년대 들어 딥러닝의 발전으로 패러다임이 크게 변화했습니다. 특히 2017년 트랜스포머 아키텍처의 등장은 자연어처리 분야에 혁명을 가져왔으며, GPT, BERT 등의 대규모 언어 모델이 등장했습니다. 최근에는 GPT-4, Claude 등 대화형 AI가 일상생활에서도 널리 활용되고 있습니다.",
        "output": "1. AI는 1950년대부터 시작된 오랜 역사를 가진 분야입니다.\n2. 2017년 트랜스포머의 등장으로 자연어처리가 혁명적으로 발전했습니다.\n3. 현재 GPT-4, Claude 등 대화형 AI가 일상에서 널리 활용되고 있습니다."
    },
    {
        "instruction": "다음 단어의 반의어를 알려주세요.",
        "input": "행복",
        "output": "'행복'의 반의어는 '불행' 또는 '슬픔'입니다."
    },
    {
        "instruction": "주어진 데이터를 표 형식으로 정리하세요.",
        "input": "사과 500원, 바나나 300원, 포도 1000원, 딸기 800원",
        "output": "| 과일 | 가격 |\n|------|------|\n| 사과 | 500원 |\n| 바나나 | 300원 |\n| 포도 | 1,000원 |\n| 딸기 | 800원 |"
    },
    {
        "instruction": "다음 코드의 버그를 찾아 수정하세요.",
        "input": "def factorial(n):\n    if n == 0:\n        return 0\n    return n * factorial(n-1)",
        "output": "버그: factorial(0)이 0을 반환합니다. 0!은 1이어야 합니다.\n\n수정된 코드:\ndef factorial(n):\n    if n == 0:\n        return 1  # 0! = 1\n    return n * factorial(n-1)"
    },
    {
        "instruction": "비즈니스 이메일을 작성하세요.",
        "input": "미팅 일정 변경 요청, 3월 20일에서 3월 25일로",
        "output": "제목: 미팅 일정 변경 요청\n\n안녕하세요,\n\n기존에 3월 20일로 예정되어 있던 미팅 일정에 대해 변경을 요청드립니다.\n\n불가피한 사정으로 인해 3월 25일(화)로 변경이 가능할지 여쭤봅니다.\n시간과 장소는 기존과 동일하게 유지하면 좋겠습니다.\n\n일정 확인 후 회신 부탁드립니다.\n\n감사합니다."
    },
    {
        "instruction": "머신러닝에서 과적합(Overfitting)이 무엇인지 설명하세요.",
        "input": "",
        "output": "과적합(Overfitting)은 머신러닝 모델이 학습 데이터에 너무 잘 맞춰져서, 새로운 데이터에 대한 예측 성능이 떨어지는 현상입니다.\n\n원인:\n1. 모델이 너무 복잡한 경우\n2. 학습 데이터가 너무 적은 경우\n3. 학습을 너무 오래 한 경우\n\n해결 방법:\n1. 더 많은 데이터 수집\n2. 정규화(Regularization) 적용\n3. 드롭아웃(Dropout) 사용\n4. 조기 종료(Early Stopping)"
    }
]

# 파일로 저장
sample_file = os.path.join(DATA_DIR, "alpaca_ko_sample.json")
with open(sample_file, 'w', encoding='utf-8') as f:
    json.dump(sample_data, f, ensure_ascii=False, indent=2)

print(f"✅ 샘플 데이터 생성 완료: {sample_file}")
print(f"   데이터 수: {len(sample_data)}개")
print(f"   형식: Alpaca (instruction, input, output)")
print(f"\n📊 카테고리 분포:")

categories = ["번역", "감성분석", "창작", "코딩", "요약", "어휘", "데이터정리", "디버깅", "비즈니스", "설명"]
for i, (cat, data) in enumerate(zip(categories, sample_data)):
    print(f"   {i+1}. [{cat}] {data['instruction'][:40]}...")

📝 실습용 샘플 데이터 생성
✅ 샘플 데이터 생성 완료: /home/yskim/LLM_Advanced/data/samples/alpaca_ko_sample.json
   데이터 수: 10개
   형식: Alpaca (instruction, input, output)

📊 카테고리 분포:
   1. [번역] 다음 문장을 영어로 번역하세요....
   2. [감성분석] 다음 텍스트의 감성을 분석하세요....
   3. [창작] 주어진 키워드로 짧은 이야기를 작성하세요....
   4. [코딩] 파이썬에서 리스트 컴프리헨션의 사용법을 설명하세요....
   5. [요약] 다음 문장을 3줄로 요약하세요....
   6. [어휘] 다음 단어의 반의어를 알려주세요....
   7. [데이터정리] 주어진 데이터를 표 형식으로 정리하세요....
   8. [디버깅] 다음 코드의 버그를 찾아 수정하세요....
   9. [비즈니스] 비즈니스 이메일을 작성하세요....
   10. [설명] 머신러닝에서 과적합(Overfitting)이 무엇인지 설명하세요....


---
## 3️⃣ 4. 데이터 정제 (Cleaning)

### 정제가 필요한 이유
- 📌 중복 데이터 → 학습 편향
- 📌 저품질 데이터 → 모델 성능 저하
- 📌 너무 짧거나 긴 데이터 → 학습 비효율
- 📌 잘못된 형식 → 학습 오류

### 주요 정제 단계

| 단계 | 설명 | 기준 |
|------|------|------|
| 형식 검증 | 필수 필드 존재 여부 | instruction, output 필수 |
| 중복 제거 | 동일/유사 데이터 제거 | 정확 매칭 + 유사도 |
| 길이 필터링 | 너무 짧거나 긴 데이터 제거 | output 10~2000자 |
| 품질 필터링 | 의미 없는 응답 제거 | 빈 응답, 반복 등 |
| 언어 필터링 | 대상 언어가 아닌 데이터 제거 | 한국어 비율 체크 |

In [5]:
# 데이터 정제 파이프라인 구현
print("🔧 데이터 정제 파이프라인 구현")
print("=" * 60)

class DataCleaner:
    """파인튜닝 데이터 정제기"""
    
    def __init__(self, min_output_len=10, max_output_len=2000,
                 min_instruction_len=5):
        self.min_output_len = min_output_len
        self.max_output_len = max_output_len
        self.min_instruction_len = min_instruction_len
        self.stats = {
            "total": 0,
            "passed": 0,
            "removed_format": 0,
            "removed_length": 0,
            "removed_quality": 0,
            "removed_duplicate": 0,
        }
    
    def validate_format(self, item):
        """형식 검증"""
        if not isinstance(item, dict):
            return False
        if "instruction" not in item or "output" not in item:
            return False
        if not item["instruction"].strip() or not item["output"].strip():
            return False
        return True
    
    def check_length(self, item):
        """길이 필터링"""
        output_len = len(item["output"])
        instruction_len = len(item["instruction"])
        
        if output_len < self.min_output_len:
            return False
        if output_len > self.max_output_len:
            return False
        if instruction_len < self.min_instruction_len:
            return False
        return True
    
    def check_quality(self, item):
        """품질 필터링"""
        output = item["output"]
        
        # 반복 텍스트 체크
        words = output.split()
        if len(words) > 5:
            unique_ratio = len(set(words)) / len(words)
            if unique_ratio < 0.3:  # 단어의 70% 이상이 반복
                return False
        
        # 의미없는 응답 체크
        low_quality_patterns = [
            "모르겠습니다", "답변할 수 없습니다", "정보가 없습니다",
            "I don't know", "I cannot", "N/A"
        ]
        for pattern in low_quality_patterns:
            if output.strip() == pattern:
                return False
        
        return True
    
    def clean(self, data):
        """전체 정제 파이프라인 실행"""
        self.stats["total"] = len(data)
        cleaned = []
        seen = set()  # 중복 체크용
        
        for item in data:
            # 1. 형식 검증
            if not self.validate_format(item):
                self.stats["removed_format"] += 1
                continue
            
            # 2. 중복 제거
            key = item["instruction"] + item["output"]
            if key in seen:
                self.stats["removed_duplicate"] += 1
                continue
            seen.add(key)
            
            # 3. 길이 필터링
            if not self.check_length(item):
                self.stats["removed_length"] += 1
                continue
            
            # 4. 품질 필터링
            if not self.check_quality(item):
                self.stats["removed_quality"] += 1
                continue
            
            cleaned.append(item)
        
        self.stats["passed"] = len(cleaned)
        return cleaned
    
    def print_report(self):
        """정제 결과 리포트 출력"""
        print(f"\n📊 데이터 정제 결과 리포트")
        print(f"{'='*50}")
        print(f"  총 데이터:     {self.stats['total']:>6d}")
        print(f"  형식 오류:     {self.stats['removed_format']:>6d}")
        print(f"  중복 제거:     {self.stats['removed_duplicate']:>6d}")
        print(f"  길이 필터링:   {self.stats['removed_length']:>6d}")
        print(f"  품질 필터링:   {self.stats['removed_quality']:>6d}")
        print(f"  {'─'*40}")
        print(f"  최종 데이터:   {self.stats['passed']:>6d} ({self.stats['passed']/self.stats['total']*100:.1f}%)")


# 테스트 데이터 (의도적으로 불량 데이터 포함)
test_data = sample_data.copy()
test_data.extend([
    # 중복 데이터
    sample_data[0],
    # 형식 오류
    {"instruction": "질문"},  # output 없음
    {"instruction": "", "input": "", "output": "답변"},  # 빈 instruction
    # 너무 짧은 응답
    {"instruction": "이것은 무엇인가요?", "input": "", "output": "네"},
    # 저품질 응답
    {"instruction": "질문입니다", "input": "", "output": "모르겠습니다"},
])

print(f"📌 테스트 데이터: {len(test_data)}개 (정상 {len(sample_data)}개 + 불량 {len(test_data)-len(sample_data)}개)")

# 정제 실행
cleaner = DataCleaner()
cleaned_data = cleaner.clean(test_data)
cleaner.print_report()

print(f"\n✅ 정제 파이프라인 동작 확인 완료!")

🔧 데이터 정제 파이프라인 구현
📌 테스트 데이터: 15개 (정상 10개 + 불량 5개)

📊 데이터 정제 결과 리포트
  총 데이터:         15
  형식 오류:          2
  중복 제거:          1
  길이 필터링:        2
  품질 필터링:        0
  ────────────────────────────────────────
  최종 데이터:       10 (66.7%)

✅ 정제 파이프라인 동작 확인 완료!


---
## 4️⃣ 5. 데이터 증강 기법

### 주요 증강 기법

| 기법 | 설명 | 적합한 경우 |
|------|------|----------|
| **Paraphrasing** | 같은 의미의 다른 표현 | 지시문 다양화 |
| **Back-Translation** | 번역 후 재번역 | 언어 다양성 |
| **Self-Instruct** | LLM으로 데이터 생성 | 데이터 부족 시 |
| **Distillation** | 큰 모델로 데이터 생성 | 품질 향상 |
| **Template Variation** | 템플릿 변경 | 형식 다양화 |

In [6]:
# 템플릿 기반 데이터 증강
print("🔧 템플릿 기반 데이터 증강 (Instruction Variation)")
print("=" * 60)

# 지시문 템플릿 변형
instruction_templates = {
    "번역": [
        "다음 문장을 영어로 번역하세요.",
        "아래 한국어를 영어로 바꿔주세요.",
        "다음을 영어로 옮겨주세요.",
        "영어 번역을 부탁드립니다.",
        "한영 번역을 해주세요.",
    ],
    "감성분석": [
        "다음 텍스트의 감성을 분석하세요.",
        "아래 글의 감정을 판단해주세요.",
        "이 문장이 긍정적인지 부정적인지 분석하세요.",
        "감성 분석을 수행해주세요.",
        "다음 텍스트의 감정 톤을 파악해주세요.",
    ],
    "요약": [
        "다음 문장을 3줄로 요약하세요.",
        "아래 내용을 간단히 요약해주세요.",
        "핵심 내용만 추려서 정리해주세요.",
        "다음 글의 요점을 정리하세요.",
        "간략한 요약을 작성해주세요.",
    ]
}

def augment_with_templates(data, templates_dict):
    """템플릿을 활용한 데이터 증강"""
    augmented = []
    
    for item in data:
        augmented.append(item)  # 원본 유지
        
        # 매칭되는 카테고리 찾기
        for category, templates in templates_dict.items():
            if any(kw in item["instruction"] for kw in ["번역", "영어로"]):
                if category == "번역":
                    for template in templates:
                        if template != item["instruction"]:
                            new_item = item.copy()
                            new_item["instruction"] = template
                            augmented.append(new_item)
                    break
            elif any(kw in item["instruction"] for kw in ["감성", "감정"]):
                if category == "감성분석":
                    for template in templates:
                        if template != item["instruction"]:
                            new_item = item.copy()
                            new_item["instruction"] = template
                            augmented.append(new_item)
                    break
    
    return augmented

augmented_data = augment_with_templates(sample_data, instruction_templates)
print(f"📌 원본 데이터: {len(sample_data)}개")
print(f"📌 증강 후 데이터: {len(augmented_data)}개")
print(f"📌 증가율: {(len(augmented_data)/len(sample_data)-1)*100:.0f}%")

# 증강된 예시 표시
print(f"\n📋 증강된 번역 데이터 예시:")
for item in augmented_data:
    if "번역" in item["instruction"] or "영어로" in item["instruction"] or "한영" in item["instruction"]:
        print(f"  🔹 {item['instruction']}")

print("\n💡 실제 파인튜닝에서는 GPT-4 등을 활용하여 더 다양하고 자연스러운 증강이 가능합니다.")

🔧 템플릿 기반 데이터 증강 (Instruction Variation)
📌 원본 데이터: 10개
📌 증강 후 데이터: 18개
📌 증가율: 80%

📋 증강된 번역 데이터 예시:
  🔹 다음 문장을 영어로 번역하세요.
  🔹 아래 한국어를 영어로 바꿔주세요.
  🔹 다음을 영어로 옮겨주세요.
  🔹 영어 번역을 부탁드립니다.
  🔹 한영 번역을 해주세요.

💡 실제 파인튜닝에서는 GPT-4 등을 활용하여 더 다양하고 자연스러운 증강이 가능합니다.


---
## 5️⃣ 6. 실습: 샘플 데이터 로드 및 변환

앞에서 생성한 `alpaca_ko_sample.json` 파일을 로드하고 다양한 형식으로 변환해봅시다.

In [7]:
import numpy as np
# 데이터 로드 및 탐색
print("📥 샘플 데이터 로드 및 탐색")
print("=" * 60)

# 파일 로드
with open(os.path.join(DATA_DIR, "alpaca_ko_sample.json"), 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"✅ 로드 완료: {len(data)}개 데이터\n")

# 데이터 통계
instruction_lens = [len(d["instruction"]) for d in data]
input_lens = [len(d.get("input", "")) for d in data]
output_lens = [len(d["output"]) for d in data]

print("📊 데이터 통계:")
print(f"  Instruction 길이: 평균 {np.mean(instruction_lens):.0f}자, "
      f"최소 {min(instruction_lens)}자, 최대 {max(instruction_lens)}자")
print(f"  Input 길이:       평균 {np.mean(input_lens):.0f}자, "
      f"최소 {min(input_lens)}자, 최대 {max(input_lens)}자")
print(f"  Output 길이:      평균 {np.mean(output_lens):.0f}자, "
      f"최소 {min(output_lens)}자, 최대 {max(output_lens)}자")

# Input이 있는/없는 데이터 비율
has_input = sum(1 for d in data if d.get("input", "").strip())
print(f"\n📊 Input 유무:")
print(f"  Input 있음: {has_input}개 ({has_input/len(data)*100:.0f}%)")
print(f"  Input 없음: {len(data)-has_input}개 ({(len(data)-has_input)/len(data)*100:.0f}%)")

📥 샘플 데이터 로드 및 탐색
✅ 로드 완료: 10개 데이터

📊 데이터 통계:
  Instruction 길이: 평균 22자, 최소 16자, 최대 36자
  Input 길이:       평균 44자, 최소 0자, 최대 239자
  Output 길이:      평균 121자, 최소 27자, 최대 241자

📊 Input 유무:
  Input 있음: 8개 (80%)
  Input 없음: 2개 (20%)


In [8]:
# Alpaca → ChatML 전체 변환
print("🔄 Alpaca → ChatML 전체 변환")
print("=" * 60)

def convert_alpaca_to_chatml(alpaca_list, system_prompt=None):
    """Alpaca 데이터 리스트를 ChatML 형식으로 변환"""
    chatml_list = []
    
    for item in alpaca_list:
        messages = []
        
        # System prompt 추가 (선택)
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        
        # User 메시지
        user_content = item["instruction"]
        if item.get("input", "").strip():
            user_content += f"\n\n{item['input']}"
        messages.append({"role": "user", "content": user_content})
        
        # Assistant 응답
        messages.append({"role": "assistant", "content": item["output"]})
        
        chatml_list.append({"messages": messages})
    
    return chatml_list

# 변환 실행
system_prompt = "당신은 유용하고 정확한 답변을 제공하는 한국어 AI 어시스턴트입니다."
chatml_data = convert_alpaca_to_chatml(data, system_prompt=system_prompt)

# 결과 확인
print(f"✅ 변환 완료: {len(chatml_data)}개\n")

print("📋 변환 결과 예시 (첫 번째 데이터):")
print(json.dumps(chatml_data[0], ensure_ascii=False, indent=2))

# ChatML 형식으로 저장
chatml_file = os.path.join(DATA_DIR, "chatml_ko_sample.json")
with open(chatml_file, 'w', encoding='utf-8') as f:
    json.dump(chatml_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ ChatML 형식으로 저장: {chatml_file}")

🔄 Alpaca → ChatML 전체 변환
✅ 변환 완료: 10개

📋 변환 결과 예시 (첫 번째 데이터):
{
  "messages": [
    {
      "role": "system",
      "content": "당신은 유용하고 정확한 답변을 제공하는 한국어 AI 어시스턴트입니다."
    },
    {
      "role": "user",
      "content": "다음 문장을 영어로 번역하세요.\n\n오늘 날씨가 매우 좋습니다."
    },
    {
      "role": "assistant",
      "content": "The weather is very nice today."
    }
  ]
}

✅ ChatML 형식으로 저장: /home/yskim/LLM_Advanced/data/samples/chatml_ko_sample.json


---
## 6️⃣ 7. HuggingFace datasets 라이브러리 활용

In [9]:
# HuggingFace datasets 라이브러리 사용법
print("📚 HuggingFace datasets 라이브러리 활용")
print("=" * 60)

try:
    from datasets import Dataset, DatasetDict, load_dataset
    print("✅ datasets 라이브러리 import 성공\n")
    
    # 1. JSON 파일에서 Dataset 생성
    print("1️⃣ 로컬 JSON 파일에서 Dataset 생성")
    print("-" * 40)
    
    # Alpaca 데이터를 Dataset으로 변환
    dataset = Dataset.from_list(data)
    print(f"   Dataset: {dataset}")
    print(f"   컬럼: {dataset.column_names}")
    print(f"   첫 번째 데이터: {dataset[0]['instruction'][:50]}...")
    
    # 2. Train/Test 분할
    print(f"\n2️⃣ Train/Test 분할")
    print("-" * 40)
    
    split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
    print(f"   Train: {len(split_dataset['train'])}개")
    print(f"   Test:  {len(split_dataset['test'])}개")
    
    # 3. 데이터 매핑 (전처리)
    print(f"\n3️⃣ 데이터 매핑 (형식 변환)")
    print("-" * 40)
    
    def format_to_prompt(example):
        """Alpaca 형식을 프롬프트 문자열로 변환"""
        prompt = f"### 지시:\n{example['instruction']}\n\n"
        if example.get('input', '').strip():
            prompt += f"### 입력:\n{example['input']}\n\n"
        prompt += f"### 응답:\n{example['output']}"
        return {"text": prompt, "length": len(prompt)}
    
    formatted_dataset = dataset.map(format_to_prompt)
    print(f"   새 컬럼 추가: {formatted_dataset.column_names}")
    print(f"\n   📋 변환 결과 예시:")
    print(f"   {formatted_dataset[0]['text'][:200]}...")
    
    # 4. 필터링
    print(f"\n4️⃣ 데이터 필터링")
    print("-" * 40)
    
    long_data = formatted_dataset.filter(lambda x: x['length'] > 200)
    print(f"   200자 이상 데이터: {len(long_data)}개 / {len(formatted_dataset)}개")

except ImportError:
    print("⚠️ datasets 라이브러리가 설치되어 있지 않습니다.")
    print("   설치: pip install datasets")
    print("\n📝 코드 예시만 확인해주세요 (설치 후 실행 가능)")

📚 HuggingFace datasets 라이브러리 활용


/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ datasets 라이브러리 import 성공

1️⃣ 로컬 JSON 파일에서 Dataset 생성
----------------------------------------
   Dataset: Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 10
})
   컬럼: ['instruction', 'input', 'output']
   첫 번째 데이터: 다음 문장을 영어로 번역하세요....

2️⃣ Train/Test 분할
----------------------------------------
   Train: 8개
   Test:  2개

3️⃣ 데이터 매핑 (형식 변환)
----------------------------------------


Map: 100%|██████████| 10/10 [00:00<00:00, 1679.33 examples/s]


   새 컬럼 추가: ['instruction', 'input', 'output', 'text', 'length']

   📋 변환 결과 예시:
   ### 지시:
다음 문장을 영어로 번역하세요.

### 입력:
오늘 날씨가 매우 좋습니다.

### 응답:
The weather is very nice today....

4️⃣ 데이터 필터링
----------------------------------------


Filter: 100%|██████████| 10/10 [00:00<00:00, 2509.91 examples/s]

   200자 이상 데이터: 5개 / 10개


In [10]:
# HuggingFace Hub에서 데이터 다운로드 예시
print("📥 HuggingFace Hub 데이터 다운로드 예시")
print("=" * 60)

print("""
📌 유용한 한국어 데이터셋 (HuggingFace Hub):

# 1. KoAlpaca (범용 Instruction)
dataset = load_dataset("beomi/KoAlpaca-v1.1a")

# 2. NSMC (감성분석)
dataset = load_dataset("nsmc")

# 3. KorQuAD (질의응답)
dataset = load_dataset("squad_kor_v1")

# 4. Open-Orca Korean
dataset = load_dataset("kyujinpy/Open-platypus-Commercial")

# 5. 특정 split만 로드
train_data = load_dataset("beomi/KoAlpaca-v1.1a", split="train")

# 6. streaming 모드 (대용량 데이터)
dataset = load_dataset("beomi/KoAlpaca-v1.1a", streaming=True)
for example in dataset['train'].take(5):
    print(example)
""")

print("💡 수강생 주의사항:")
print("  - 25명이 동시에 HuggingFace에서 다운로드하면 느려질 수 있습니다")
print("  - 미리 다운받은 데이터를 NAS/USB로 배포받으세요")
print("  - HF_HOME 환경변수로 캐시 경로를 공유 폴더로 설정 가능")

📥 HuggingFace Hub 데이터 다운로드 예시

📌 유용한 한국어 데이터셋 (HuggingFace Hub):

# 1. KoAlpaca (범용 Instruction)
dataset = load_dataset("beomi/KoAlpaca-v1.1a")

# 2. NSMC (감성분석)
dataset = load_dataset("nsmc")

# 3. KorQuAD (질의응답)
dataset = load_dataset("squad_kor_v1")

# 4. Open-Orca Korean
dataset = load_dataset("kyujinpy/Open-platypus-Commercial")

# 5. 특정 split만 로드
train_data = load_dataset("beomi/KoAlpaca-v1.1a", split="train")

# 6. streaming 모드 (대용량 데이터)
dataset = load_dataset("beomi/KoAlpaca-v1.1a", streaming=True)
for example in dataset['train'].take(5):
    print(example)

💡 수강생 주의사항:
  - 25명이 동시에 HuggingFace에서 다운로드하면 느려질 수 있습니다
  - 미리 다운받은 데이터를 NAS/USB로 배포받으세요
  - HF_HOME 환경변수로 캐시 경로를 공유 폴더로 설정 가능


---
## 7️⃣ 8. 데이터 품질 검증 체크리스트

파인튜닝 전 반드시 확인해야 할 데이터 품질 항목들입니다.

In [11]:
# 데이터 품질 검증 도구
print("✅ 데이터 품질 검증 체크리스트")
print("=" * 60)

def validate_dataset(data, format_type="alpaca"):
    """데이터셋 품질 검증"""
    checks = []
    
    # 1. 데이터 수 확인
    n = len(data)
    checks.append({
        "check": "데이터 수",
        "result": f"{n}개",
        "status": "✅" if n >= 100 else "⚠️" if n >= 10 else "❌",
        "note": "최소 100개 이상 권장, 1000개 이상이 이상적"
    })
    
    # 2. 형식 검증
    if format_type == "alpaca":
        valid_format = sum(1 for d in data if "instruction" in d and "output" in d)
    else:
        valid_format = sum(1 for d in data if "messages" in d)
    
    checks.append({
        "check": "형식 일관성",
        "result": f"{valid_format}/{n} ({valid_format/n*100:.0f}%)",
        "status": "✅" if valid_format == n else "❌",
        "note": "100% 필수"
    })
    
    # 3. 중복 확인
    unique_instructions = len(set(d.get("instruction", "") for d in data))
    dup_ratio = (n - unique_instructions) / n * 100
    checks.append({
        "check": "중복 비율",
        "result": f"{dup_ratio:.1f}%",
        "status": "✅" if dup_ratio < 5 else "⚠️" if dup_ratio < 20 else "❌",
        "note": "5% 미만 권장"
    })
    
    # 4. 빈 필드 확인
    empty_output = sum(1 for d in data if not d.get("output", "").strip())
    checks.append({
        "check": "빈 응답",
        "result": f"{empty_output}개",
        "status": "✅" if empty_output == 0 else "❌",
        "note": "0개 필수"
    })
    
    # 5. 길이 분포
    output_lens = [len(d.get("output", "")) for d in data]
    avg_len = np.mean(output_lens)
    checks.append({
        "check": "평균 응답 길이",
        "result": f"{avg_len:.0f}자",
        "status": "✅" if 50 < avg_len < 1000 else "⚠️",
        "note": "50~1000자 권장"
    })
    
    # 6. 극단값 확인
    too_short = sum(1 for l in output_lens if l < 10)
    too_long = sum(1 for l in output_lens if l > 2000)
    checks.append({
        "check": "극단값 (너무 짧음)",
        "result": f"{too_short}개 (<10자)",
        "status": "✅" if too_short == 0 else "⚠️",
        "note": "제거 권장"
    })
    checks.append({
        "check": "극단값 (너무 김)",
        "result": f"{too_long}개 (>2000자)",
        "status": "✅" if too_long == 0 else "⚠️",
        "note": "max_seq_length 초과 시 잘림"
    })
    
    # 리포트 출력
    print(f"\n{'체크 항목':>20s} | {'결과':>20s} | {'상태':>4s} | {'기준/참고'}")
    print("-" * 80)
    for c in checks:
        print(f"{c['check']:>20s} | {c['result']:>20s} | {c['status']:>4s} | {c['note']}")
    
    passed = sum(1 for c in checks if c['status'] == '✅')
    total = len(checks)
    print(f"\n📊 검증 결과: {passed}/{total} 통과")
    
    return checks

# 검증 실행
validate_dataset(data, format_type="alpaca")

✅ 데이터 품질 검증 체크리스트

               체크 항목 |                   결과 |   상태 | 기준/참고
--------------------------------------------------------------------------------
               데이터 수 |                  10개 |   ⚠️ | 최소 100개 이상 권장, 1000개 이상이 이상적
              형식 일관성 |         10/10 (100%) |    ✅ | 100% 필수
               중복 비율 |                 0.0% |    ✅ | 5% 미만 권장
                빈 응답 |                   0개 |    ✅ | 0개 필수
            평균 응답 길이 |                 121자 |    ✅ | 50~1000자 권장
         극단값 (너무 짧음) |            0개 (<10자) |    ✅ | 제거 권장
          극단값 (너무 김) |          0개 (>2000자) |    ✅ | max_seq_length 초과 시 잘림

📊 검증 결과: 6/7 통과


[{'check': '데이터 수',
  'result': '10개',
  'status': '⚠️',
  'note': '최소 100개 이상 권장, 1000개 이상이 이상적'},
 {'check': '형식 일관성',
  'result': '10/10 (100%)',
  'status': '✅',
  'note': '100% 필수'},
 {'check': '중복 비율', 'result': '0.0%', 'status': '✅', 'note': '5% 미만 권장'},
 {'check': '빈 응답', 'result': '0개', 'status': '✅', 'note': '0개 필수'},
 {'check': '평균 응답 길이', 'result': '121자', 'status': '✅', 'note': '50~1000자 권장'},
 {'check': '극단값 (너무 짧음)',
  'result': '0개 (<10자)',
  'status': '✅',
  'note': '제거 권장'},
 {'check': '극단값 (너무 김)',
  'result': '0개 (>2000자)',
  'status': '✅',
  'note': 'max_seq_length 초과 시 잘림'}]

---
## 📝 9. 정리 및 핵심 요약

### 🎯 이번 세션에서 배운 핵심 개념

| # | 개념 | 핵심 내용 |
|---|------|----------|
| 1️⃣ | **데이터 형식** | Alpaca(단순), ShareGPT(멀티턴), ChatML(표준) |
| 2️⃣ | **데이터 수집** | 공개 데이터, 합성 데이터, 직접 제작 |
| 3️⃣ | **데이터 정제** | 형식 검증, 중복 제거, 길이/품질 필터링 |
| 4️⃣ | **데이터 증강** | 템플릿 변형, Self-Instruct, Distillation |
| 5️⃣ | **형식 변환** | Alpaca ↔ ChatML 변환 함수 구현 |
| 6️⃣ | **HF datasets** | load_dataset, map, filter, train_test_split |
| 7️⃣ | **품질 검증** | 체크리스트 기반 자동 검증 |

### 🔑 실무 핵심 원칙

```
📌 데이터 품질 > 데이터 양
📌 ChatML 형식으로 통일 권장
📌 전체 시간의 70%를 데이터 준비에 투자
📌 최소 100개, 이상적으로 1,000~10,000개
📌 정제 파이프라인을 자동화하여 재사용
```

### 📚 다음 세션 예고
- **Session 15**: 합성 데이터 생성 & Distillation - GPT-4를 활용한 데이터 자동 생성

In [12]:
print("\n✅ Session 14 완료!")
print("📚 다음 세션: 합성 데이터 생성 & Distillation")
print(f"\n📁 생성된 파일:")
print(f"  1. {os.path.join(DATA_DIR, 'alpaca_ko_sample.json')} (Alpaca 형식)")
print(f"  2. {os.path.join(DATA_DIR, 'chatml_ko_sample.json')} (ChatML 형식)")
print("\n💡 이 데이터는 이후 실습 노트북에서 계속 사용됩니다!")


✅ Session 14 완료!
📚 다음 세션: 합성 데이터 생성 & Distillation

📁 생성된 파일:
  1. /home/yskim/LLM_Advanced/data/samples/alpaca_ko_sample.json (Alpaca 형식)
  2. /home/yskim/LLM_Advanced/data/samples/chatml_ko_sample.json (ChatML 형식)

💡 이 데이터는 이후 실습 노트북에서 계속 사용됩니다!
